# 02 · What makes the trajectory features work?

**NFL Big Data Bowl 2026 — Prediction · Feature research completion gate**

The bank contains **7,999 candidates in 20 families**. The central question is whether their signals improve the official metric under the same estimator and honest chronological validation. Follow the evidence from a 64-feature reference through nested additions, wider budgets, family removals, and availability tests.

The final section reports the sealed 48-game holdout evaluation separately from development research. Neither is a leaderboard claim. The final section retains an owner-controlled export; automated review never enables it.

**Research milestone:** all 15 feature-gate criteria now pass. The feature definitions and refit columns are frozen. Both final models have been refitted on 224 games. The final section distinguishes this completed fit from the separately sealed reserved evaluation.

In [ ]:
import io
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from IPython.display import Image, Markdown, display

from nfl_trajectory.research import load_evidence, load_research_report

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
PUBLISHED = ROOT / "docs/results"
feature_summary, selection, evidence_label = load_evidence(ROOT)


def extra_report(local, published):
    path = ROOT / "artifacts" / local
    if not path.is_file():
        path = PUBLISHED / published
    if not path.is_file():
        return None
    result = json.loads(path.read_text())
    if result.get("status") != "passed" or result.get("holdout_evaluation") != "not_run":
        raise ValueError("A completed report with an unscored holdout is required.")
    return result


def static(figure):
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    plt.close(figure)
    display(Image(data=buffer.getvalue()))


def interactive(figure):
    figure.update_layout(template="plotly_white", font={"size": 13})
    display({"application/vnd.plotly.v1+json": json.loads(figure.to_json())}, raw=True)


research = importance = None
if (ROOT / "artifacts/research/report/summary.json").is_file() or (
    PUBLISHED / "feature_research.json"
).is_file():
    research, importance, research_label = load_research_report(ROOT)
    display(Markdown(f"**Research evidence:** {research_label}"))
probe = extra_report("nonlinear_probe/summary.json", "feature_probe.json")
ablation = extra_report("feature_ablation/summary.json", "feature_ablation.json")
budget = extra_report("feature_budget/summary.json", "feature_budget.json")
joint = extra_report("joint_linear/summary.json", "feature_joint.json")
inference = extra_report("research/inference/summary.json", "feature_inference.json")
input_failure = extra_report(
    "research/inference_before_fallback.json", "feature_input_failure.json"
)
attribution = extra_report("feature_attribution/summary.json", "feature_attribution.json")
gateway = extra_report("research/gateway/summary.json", "feature_gateway.json")
tree = extra_report("research/tree/summary.json", "feature_tree.json")
wide_ablation = extra_report("wide_ablation/summary.json", "feature_wide_ablation.json")
simplification = extra_report("simplification/summary.json", "feature_simplification.json")
gate = extra_report("research/gate/summary.json", "feature_gate.json")
diagnostics = extra_report("research/gate/diagnostics.json", "feature_diagnostics.json")
display(
    pd.DataFrame(
        [
            {"Evidence": name, "Available": value is not None}
            for name, value in [
                ("Candidate research", research),
                ("Fixed estimator comparisons", probe),
                ("Strict removals", ablation),
                ("Width search", budget),
                ("Joint linear fit", joint),
                ("Raw inference", inference),
                ("Wide attribution", attribution),
                ("Gateway", gateway),
                ("Portable tree", tree),
                ("Current wide group refits", wide_ablation),
                ("Combined feature omission", simplification),
                ("Completion decision", gate),
            ]
        ]
    )
)
if tree and diagnostics:
    display(
        Markdown(
            "**Verified shallow tree predictor** · "
            f"**{tree['retained_features']:,} actually used features** · "
            f"**{diagnostics['metrics']['coordinate_rmse_yards']:.5f} coordinate RMSE**"
        )
    )
    display(pd.Series(diagnostics["metrics"], name="Current development diagnostics").round(5))
if gate:
    display(Markdown(f"**Feature completion gate: {gate['feature_gate'].upper()}**"))
if diagnostics:
    comparison = diagnostics["controlled_comparison"]
    display(
        Markdown(
            f"**Feature improvement with estimator settings fixed: "
            f"{comparison['feature_gain_percent']:.2f}%.** "
            f"Landing features: {comparison['baseline_coordinate_rmse_yards']:.5f} RMSE; "
            f"engineered representation: {comparison['engineered_coordinate_rmse_yards']:.5f}."
        )
    )
    display(
        pd.Series(
            comparison["paired_delta_ci95_yards"],
            index=["lower", "upper"],
            name="Paired 95% interval for RMSE change (yards)",
        )
    )

## 1 · Attribute gains to the features with estimator settings fixed

Every nonlinear comparison uses two residual regressors: **100 iterations, depth 4, at most 15 leaves, learning rate 0.07, minimum leaf size 60, L2 penalty 1, and 63 bins**. Early stopping is disabled so a random frame split cannot enter training. The physical baseline is refitted within each training fold.

The landing reference, engineered core, context, representations, and their union use identical rows and settings. Independent noise columns are a negative control and never enter feature selection. Comparing these rows isolates representation gains at this fixed capacity; comparing a linear score with a tree score would also change the estimator.

In [ ]:
if probe:
    scores = pd.DataFrame(probe["models"])
    scores = scores.loc[~scores.model.eq("constant_velocity")].copy()
    reference = scores.loc[scores.model.eq("landing_features"), "coordinate_rmse_yards"].iloc[0]
    scores["feature_gain_percent"] = 100 * (1 - scores.coordinate_rmse_yards / reference)
    labels = {
        "landing_features": "Landing reference",
        "engineered_core": "Engineered motion and geometry",
        "core_plus_noise": "Core plus independent noise",
        "core_plus_context": "Core plus contextual features",
        "core_plus_representation": "Core plus learned/geometric representations",
        "all_engineered": "Combined engineered representation",
    }
    scores["representation"] = scores.model.map(labels)
    display(
        scores[
            [
                "representation",
                "feature_count",
                "coordinate_rmse_yards",
                "feature_gain_percent",
                "delta_vs_landing_ci95",
            ]
        ].round(5)
    )
    interactive(
        px.bar(
            scores,
            x="coordinate_rmse_yards",
            y="representation",
            orientation="h",
            color="feature_gain_percent",
            title="Same estimator, different features",
            labels={"coordinate_rmse_yards": "Development RMSE (yards)"},
        )
    )
    fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
    bars = ax.barh(scores.representation, scores.coordinate_rmse_yards, color="#247A89")
    ax.bar_label(bars, fmt="%.5f", padding=4, fontsize=9)
    ax.set_xlim(0, scores.coordinate_rmse_yards.max() * 1.11)
    ax.set_xlabel("Development coordinate RMSE (yards); lower is better")
    static(fig)
    display(Markdown(f"**Selected on inner folds:** {probe['selection']['selected_model']}"))
else:
    display(Markdown("The completed fixed-estimator report is not available in this checkout."))

## 2 · Does additional width still pay?

All candidates are screened on training data. The wider search preserves the 250-feature union, interleaves ranked families, and examines the full training-eligible pool. Additional columns pass a correlation check on 8,192 deterministic training rows at a 0.9995 threshold. The 512, 1,024, 2,048, 4,096, and 8,192 requested budgets are nested up to the eligible count.

A large candidate count is not a stopping argument. Examine incremental gains across all inner folds, paired development-game uncertainty, and computational cost. Selection pools squared errors over the three inner folds; it does not average fold RMSEs or select on development.

In [ ]:
if budget:
    widths = pd.DataFrame(
        [
            {
                "fold": fold["fold"]["name"],
                "model": row["model"],
                "feature_count": row["feature_count"],
                "coordinate_rmse_yards": row["coordinate_rmse_yards"],
            }
            for fold in [*budget["inner_folds"], budget]
            for row in fold["models"]
        ]
    )
    display(widths.pivot(index="model", columns="fold", values="coordinate_rmse_yards").round(5))
    display(pd.Series(budget["inner_scores"], name="Pooled inner-fold RMSE").sort_values())
    interactive(
        px.line(
            widths,
            x="feature_count",
            y="coordinate_rmse_yards",
            color="fold",
            markers=True,
            title="Marginal value of a wider screened representation",
            labels={"feature_count": "Retained features", "coordinate_rmse_yards": "RMSE (yards)"},
        )
    )
    fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
    for name, rows in widths.groupby("fold", sort=True):
        ax.plot(rows.feature_count, rows.coordinate_rmse_yards, marker="o", label=name)
    ax.set_xlabel("Retained features")
    ax.set_ylabel("Coordinate RMSE (yards)")
    ax.legend()
    static(fig)
    if attribution:
        display(pd.DataFrame(attribution["development"]["incremental_width"]).round(5))
else:
    display(Markdown("Width comparisons are incomplete until every fold has a verified result."))

## 3 · Which families help, and which disappoint?

Strict removals delete a family from the 250-feature union, refit with identical settings, and add no replacement columns. A positive RMSE change indicates useful conditional information. The positional linear fallback uses another estimator and is reported separately.

Reachability and role-specific destination geometry were consistently useful in the chronological experiments. Route descriptors, role-gated terms, and metadata showed more mixed or small effects. A weak sequential linear correction can still help a nonlinear model; every conclusion must identify its comparator.

In [ ]:
if ablation:
    rows = []
    for fold in [*ablation["inner_folds"], ablation]:
        reference = next(
            r["coordinate_rmse_yards"] for r in fold["models"] if r["model"] == "all_engineered"
        )
        for row in fold["models"]:
            if row["model"].startswith("without_"):
                rows.append(
                    {
                        "fold": fold["fold"]["name"],
                        "removed": row["model"].removeprefix("without_"),
                        "rmse_change": row["coordinate_rmse_yards"] - reference,
                    }
                )
    removals = pd.DataFrame(rows)
    display(removals.pivot(index="removed", columns="fold", values="rmse_change").round(5))
    interactive(
        px.bar(
            removals,
            x="removed",
            y="rmse_change",
            color="fold",
            barmode="group",
            title="Strict family removals; fixed estimator",
            labels={"rmse_change": "RMSE increase after removal (yards)"},
        )
    )
    display(
        pd.DataFrame(ablation["models"])[["model", "feature_count", "coordinate_rmse_yards"]].round(
            5
        )
    )

### Explain the wider representation separately

The permutation study describes the full screened budget with available metadata. The availability-profile comparisons then test whether metadata or optional telemetry is needed. The next section refits removals from the selected metadata-free inference profile.

Family permutations shuffle complete trajectories within role and forecast horizon, preserve exact forecast-frame alignment, and hold the physical baseline fixed. Five seeds measure shuffle sensitivity. Their range is **not a confidence interval**. Permutation measures reliance by this model, not causal importance; correlated families may substitute for each other.

Fresh fixed-capacity refits remove body/position metadata and then all optional metadata/telemetry dependencies. Historical residual priors remain eligible in both profiles; a separate raw-input stress test clears their lookups. These checks distinguish refitted availability profiles from cold-history behavior.

In [ ]:
if attribution:
    display(
        Markdown(
            f"**Inner-fold choice:** {attribution['selected_model']} · "
            f"{attribution['development']['feature_count']:,} retained columns"
        )
    )
    dependence = pd.DataFrame(attribution["development"]["permutation"])
    display(
        dependence.drop(columns="seed_changes")
        .sort_values("rmse_increase_mean", ascending=False)
        .round(5)
    )
    interactive(
        px.bar(
            dependence.sort_values("rmse_increase_mean"),
            x="rmse_increase_mean",
            y="family",
            orientation="h",
            title="Conditional reliance of the selected wide representation",
            labels={"rmse_increase_mean": "Development RMSE increase (yards)"},
        )
    )
    figure = ROOT / "artifacts/feature_attribution/figure.png"
    if not figure.exists():
        figure = PUBLISHED / "feature_attribution.png"
    if figure.exists():
        display(Image(filename=str(figure)))
    display(
        pd.DataFrame(
            [
                {
                    "fold": fold["fold"],
                    **{
                        key: row[key]
                        for key in [
                            "model",
                            "feature_count",
                            "coordinate_rmse_yards",
                            "delta_vs_full_ci95",
                        ]
                    },
                }
                for fold in [*attribution["inner_folds"], attribution["development"]]
                for row in fold["omissions"]
            ]
        ).round(5)
    )

### Refit group removals on the current wide predictor

These refits use the selected wide availability profile before lossless pruning. Twelve disjoint groups cover all 20 feature families. Each removal preserves the physical baseline and estimator settings and adds no replacement columns. An already absent group is explicitly marked as a structural control, with no fabricated fit.

These are column-group ablations: derived information in other families can remain. Dropping direct history summaries, for example, does not remove every forecast interaction derived from motion.

Positive RMSE changes mean the group supplies useful conditional information. Negative changes identify possible simplifications. Group effects are not additive because other features can substitute. The [predeclared protocol](../docs/WIDE_ABLATION_PROTOCOL.md) keeps the gate open if an omission improves pooled inner RMSE by at least 0.5% without costing any fold more than 1%. Development intervals describe paired game uncertainty and do not choose the representation.


In [ ]:
if wide_ablation:
    wide_rows = pd.DataFrame(
        [
            {"fold": fold["fold"], **row}
            for fold in [*wide_ablation["inner_folds"], wide_ablation["development"]]
            for row in fold["models"]
        ]
    )
    display(wide_rows.pivot(index="group", columns="fold", values="rmse_change").round(5))
    display(pd.DataFrame(wide_ablation["pooled_comparisons"]).round(5))
    development_removals = wide_rows[wide_rows.fold.eq("development")].copy()
    display(
        development_removals[
            ["group", "removed_features", "refitted", "coordinate_rmse_yards", "paired_game_ci95"]
        ].round(5)
    )
    interactive(
        px.bar(
            wide_rows,
            x="group",
            y="rmse_change",
            color="fold",
            barmode="group",
            title="Strict removals from the actual wide availability profile",
            labels={"rmse_change": "RMSE change after removal (yards)"},
        )
    )
    ordered = development_removals.sort_values("rmse_change")
    fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
    positions = range(len(ordered))
    ax.hlines(
        positions,
        [bounds[0] for bounds in ordered.paired_game_ci95],
        [bounds[1] for bounds in ordered.paired_game_ci95],
        color="#64737A",
        linewidth=1.5,
    )
    ax.scatter(ordered.rmse_change, positions, color="#247A89", zorder=3)
    ax.set_yticks(positions, labels=ordered.group.str.replace("_", " "))
    ax.axvline(0, color="#444444", linewidth=0.8)
    ax.set_xlabel("Change in development RMSE (yards); paired 95% game intervals")
    static(fig)
else:
    display(
        Markdown("Current wide-profile group refits are still required before feature closure.")
    )

### Test whether two small simplifications work together

Removing direct observed-history columns and removing generic forecast crosses each slightly improved all three inner folds. Their individual gains were below the recorded 0.5% threshold. One additional experiment removes both groups together, preserving every other column and the fixed estimator settings.

This tests whether the small gains accumulate. It does not remove all motion or horizon information: derived features in the remaining families still express both. The combined omission keeps the gate open if pooled inner RMSE improves by at least 0.5% and no inner fold costs more than 1%. Development remains descriptive. See the committed [protocol](../docs/SIMPLIFICATION_PROTOCOL.md).


The completed combined omission changes pooled inner RMSE from **0.7026010 to 0.7014072**, a **0.170% improvement**. Fold changes are −0.298%, +0.324%, and −0.399%. The gain is below the predeclared threshold and is not consistent across time, so it does not justify reopening the feature search.

In [ ]:
if simplification:
    comparison_rows = []
    for fold in [*simplification["inner_folds"], simplification["development"]]:
        before = fold["reference"]["coordinate_rmse_yards"]
        after = fold["metrics"]["coordinate_rmse_yards"]
        comparison_rows.append(
            {
                "fold": fold["fold"],
                "parent_columns": fold["parent_feature_count"],
                "retained_columns": fold["feature_count"],
                "parent_rmse": before,
                "combined_omission_rmse": after,
                "rmse_change": after - before,
                "paired_game_ci95": fold["paired_game_ci95"],
            }
        )
    display(pd.DataFrame(comparison_rows).round(5))
    display(pd.Series(simplification["decision"], name="Combined-omission decision"))
else:
    display(Markdown("The targeted combined-omission test is required before feature closure."))

## 4 · Broad feature coverage with explicit prediction-time boundaries

The catalogs cover motion, landing geometry, lags, multiscale trends and volatility, robust path distributions, matched opponent/receiver histories, relative ranks, nonlinear and role-conditioned crosses, arrival feasibility, geometric player-set pools, role-specific destinations, and training-only route representations.

Player/role residual histories use **strictly earlier game dates**: the entire current date is excluded, smoothing supports sparse players, and evaluation histories are frozen from training. Route components and prototypes are fitted separately inside each training fold. Post-throw coordinates, future game results, full-season target averages, and unverified external ratings cannot enter prediction inputs.

In [ ]:
if research:
    families = pd.DataFrame(research["families"])
    display(families[["stage", "family", "candidates", "nonconstant", "constant_or_near_constant"]])
    display(
        Markdown(
            f"**{research['candidate_features']:,} candidates; "
            f"{int(families.nonconstant.sum()):,} nonconstant** on development-training rows. "
            "Eligibility alone does not prove predictive value."
        )
    )
    interactive(
        px.bar(
            families,
            x="candidates",
            y="family",
            color="stage",
            orientation="h",
            title="Candidate families and search coverage",
        )
    )
    display(
        pd.DataFrame(
            [
                {
                    "stage": row["stage"],
                    "model": row["model"],
                    "all_three_folds": len(row["all_three_folds"]),
                    "any_fold": row["selected_in_any_fold"],
                    "pairwise_jaccard": row["pairwise_jaccard"],
                }
                for row in research["stability"]
            ]
        )
    )

### Stability can be strong at the family level and weak at the column level

Combined context improved all three chronological folds. Individual selected columns overlap less because many lagged and gated variables express related signals. We report that overlap. PCA names do not guarantee identical axes across folds. This study has one labelled season: it does not establish across-season stability.

The sequential 186-feature predictor is an interpretable intermediate. Its permutation report explains that fit, not the current joint linear model or the wider tree.

In [ ]:
if research:
    fold_results = pd.DataFrame(research["fold_results"])
    chosen = fold_results.loc[
        ((fold_results.stage == "research") & (fold_results.model == "plus_balanced"))
        | ((fold_results.stage == "context") & (fold_results.model == "plus_context"))
        | ((fold_results.stage == "representation") & (fold_results.model == "plus_representation"))
    ]
    display(
        chosen[
            [
                "stage",
                "fold",
                "training_games",
                "validation_games",
                "coordinate_rmse_yards",
                "improvement_vs_parent_percent",
            ]
        ].round(5)
    )
    if importance:
        display(
            pd.DataFrame(importance["rows"])[
                ["family", "retained_features", "mean_rmse_increase_yards", "shuffle_sd_yards"]
            ].round(5)
        )
if gate:
    stability = gate["used_feature_stability"]
    display(
        Markdown(
            f"**Actual tree use:** {stability['count_used_in_all_inner_fits']:,} definitions "
            "appear in all three inner fits. This differs from merely surviving a broad screen."
        )
    )
    display(pd.DataFrame(stability["pairwise_jaccard"]).round(3))
    display(Markdown(stability["caveat"]))

## 5 · Carry the validated features into actual inference

The joint linear experiment first tests whether refitting all coefficients recovers signal lost by sequential corrections. The portable tree handoff then reuses the already fitted, fixed-capacity feature trees. It chooses between metadata-free availability profiles on the inner folds, converts their numerical splits to a standalone representation, and drops columns never used by a split. Prediction parity is required after this lossless pruning. No additional model tuning occurs.

Raw inference must reproduce the cached-feature score on **every development frame**. Stress scenarios remove metadata, remove telemetry, and clear player history. Missing required fields select an independently fitted compatible profile. Silent zero filling is not a valid availability fallback.

In [ ]:
if joint:
    display(
        pd.DataFrame(joint["models"])[["model", "feature_count", "coordinate_rmse_yards"]].round(5)
    )
    display(pd.Series(joint["inner_scores"], name="Pooled inner-fold RMSE").sort_values())
    display(Markdown(f"**Selected linear profile:** {joint['selected_model']}"))
if inference:
    display(pd.DataFrame(inference["scenarios"]).round(5))
    display(
        Markdown(
            f"**Actual raw-input predictor:** {inference['selected_stage']} / "
            f"{inference['selected_model']} · {inference['retained_features']} features. "
            "Publication verifies source hashes, fitted artifacts, and per-week replay receipts."
        )
    )
else:
    display(Markdown("Full raw-input replay must pass before this path is called validated."))
if input_failure and inference:
    before = pd.DataFrame(input_failure["scenarios"]).set_index("scenario")
    after = pd.DataFrame(inference["scenarios"]).set_index("scenario")
    display(Markdown("**Archived availability failure and the current validated remedy**"))
    display(
        pd.DataFrame(
            {
                "Archived sequential profile RMSE": before.coordinate_rmse_yards,
                "Current profile RMSE": after.coordinate_rmse_yards,
            }
        ).round(5)
    )
    display(
        Markdown(
            "The archived 186-feature profile failed badly when telemetry disappeared. "
            "That result motivated independently fitted availability profiles. "
            "This historical comparison changes both representation and estimator; "
            "the fixed-estimator experiments above establish feature attribution."
        )
    )

### Inspect one play without selecting a flattering example

The example is the first development play by game/play identifier, showing up to three requested players in identifier order. Selection does not use prediction error. Solid dots mark the throw. The embedded figure uses the current validated research predictor. Regenerating this particular figure requires private tracking; the full quantitative comparisons above also work from the published aggregates.

The physical-baseline curve shows what the learned correction changes. The per-player table can show improvements or regressions; this single example does not replace the full-game attribution evidence.


In [ ]:
from nfl_trajectory.research_visuals import load_development_example

example = load_development_example(ROOT)
if example:
    static(example.static_figure)
    interactive(example.interactive_figure)
    display(example.roles)
    display(example.metrics.round(5))
    display(
        Markdown(
            f"**Displayed predictor:** {example.selected_model} · "
            f"first development game/play: {example.game_id}/{example.play_id}."
        )
    )
else:
    display(Markdown("The canonical executed notebook contains the private-data example figure."))

### Verify the organizer interface on its own unlabelled sample

The organizer sends one play at a time and expects finite x/y predictions in the requested order. The test uses checksum-verified organizer source and sample inputs, checks package/standalone parity for every callback, and verifies Parquet row identifiers. Output stays in the isolated quality directory.

A pass establishes interface compatibility. It gives no accuracy score, across-season performance result, or competition submission.

In [ ]:
if gateway:
    display(
        pd.DataFrame(
            [
                {
                    key: gateway[key]
                    for key in [
                        "official_gateway_status",
                        "selected_model",
                        "retained_features",
                        "sample_rows",
                        "plays",
                        "sample_calendar_years",
                        "callback_seconds_median",
                        "callback_seconds_p95",
                        "callback_seconds_max",
                        "standalone_prediction_parity",
                        "labels_available",
                        "competition_submission",
                    ]
                }
            ]
        )
    )
else:
    display(Markdown("Organizer gateway validation is not yet recorded in this checkout."))

### Diagnose the current predictor by role and requested horizon

These slices use the **same portable predictor that passed raw replay and the gateway**. ADE measures displacement throughout the trajectory; FDE measures its final requested frame. Horizons group complete trajectories, preserving the meaning of FDE. Error shares show where remaining squared error is concentrated. These are descriptive development diagnostics, with no holdout or subgroup tuning.

In [ ]:
if diagnostics:
    slices = pd.DataFrame(diagnostics["slices"])
    display(
        slices[
            [
                "dimension",
                "value",
                "rows",
                "games",
                "coordinate_rmse_yards",
                "ade_frame_weighted_yards",
                "fde_trajectory_weighted_yards",
                "p95_displacement_yards",
                "squared_error_share_percent",
            ]
        ].round(4)
    )
    chart = px.bar(
        slices,
        x="value",
        y="coordinate_rmse_yards",
        facet_col="dimension",
        color="dimension",
        labels={"value": "", "coordinate_rmse_yards": "Coordinate RMSE (yards)"},
        title="Remaining error in the current validated representation",
    )
    chart.update_xaxes(matches=None)
    interactive(chart)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    for ax, dimension in zip(axes, ["role", "requested_horizon"], strict=True):
        part = slices.loc[slices.dimension.eq(dimension)]
        ax.bar(part.value, part.coordinate_rmse_yards, color="#247A89")
        ax.set_title(dimension.replace("_", " ").capitalize())
        ax.set_ylabel("Coordinate RMSE (yards)")
        ax.tick_params(axis="x", labelrotation=20)
    static(fig)

## 6 · Preserve the original research lesson

Landing ridge initially reached 0.9269 RMSE versus 0.9896 for role-conditioned motion. The broader interaction challenger replaced 23 of the 64 landing columns. Its weaker score confounded adding interactions with removing useful terms. This motivated nested additions, chronological inner selection, and strict removals.

These original diagnostics explain the research design; they are not the latest performance claim.

In [ ]:
from nfl_trajectory.research import error_budget

display(
    pd.DataFrame(feature_summary["models"])[
        [
            "model",
            "selected_features",
            "coordinate_rmse_yards",
            "ade_frame_weighted_yards",
            "fde_trajectory_weighted_yards",
            "p95_displacement_yards",
        ]
    ].round(4)
)
display(
    pd.DataFrame(
        {
            "Removed landing feature": selection["removed_from_landing"],
            "Added interaction feature": selection["added_by_interaction"],
        }
    )
)
roles = error_budget(feature_summary, "role")
horizons = error_budget(feature_summary, "forecast_second")
display(roles.round(3))
display(horizons.round(3))
interactive(
    px.bar(
        horizons,
        x="value",
        y="squared_error_share_percent",
        title="Original landing-model error budget",
        labels={
            "value": "Forecast second",
            "squared_error_share_percent": "Squared error share (%)",
        },
    )
)

## 7 · Feature engineering is a completion gate

The official metric is

$$\mathrm{RMSE}=\sqrt{\frac{\sum_{i=1}^{N}[(\hat{x}_i-x_i)^2+(\hat{y}_i-y_i)^2]}{2N}}.$$

Frames have equal coordinate weight. Game-cluster bootstraps preserve within-game dependence. Paired intervals describe uncertainty on these development games; they do not erase repeated research decisions or establish multi-season generalization.

Closing the gate requires justified coverage, training-only screening, leakage tests, fixed-estimator improvements, ablations, stability, robust inference, and diminished marginal gains from realistic extensions. External team/coaching ratings lack a verified prediction-time join here. Train-only route representations are implemented; a pretrained football foundation model is not.

The decision below applies the tolerances recorded in the [research plan](../docs/RESEARCH_PLAN.md) before full-pool results were reviewed. Any failed criterion keeps the gate open. This is a bounded research stopping decision, not a claim that all possible football representations have been invented. Final refitting and the reserved holdout come afterwards. The inference report identifies the actual exported profile and its verified metric; promotion requires raw and standalone parity.

**Closure decision:** the complete group-refit and combined-omission evidence now satisfies every recorded criterion. The final width increment improves pooled inner RMSE by only **0.071%**; the targeted combined omission improves it by **0.170%**, both with mixed folds. This supports diminishing returns within the stated search. The frozen manifest preserves **6,308 refit columns**, while the current trees use **953 inputs**. A future refit must start from the former and receive new validation evidence.

In [ ]:
if gate:
    display(pd.DataFrame(gate["checks"]))
    display(pd.DataFrame(gate["coverage"]))
    display(pd.Series(gate["measurements"], name="Stopping evidence"))
    display(
        Markdown(
            f"**Final training ready: {gate['final_training_ready']}.** "
            "The reserved holdout remains unscored in this feature research record."
        )
    )
else:
    display(Markdown("**Gate open:** current completion evidence has not been published."))

## 8 · Final refit and reserved evaluation

The final fit preserves the frozen estimator and feature schemas while incorporating the original training and development games. It refits every learned preprocessing component on those 224 games. All four coordinate fits are complete. Portable export agrees with the original estimators on every training row; this is a conversion check, not an accuracy estimate.

The reserved evaluation uses 48 later games. All forecast scenarios and reference baselines are generated and sealed before outcome access. The notebook verifies the published fit, inference, seal and evaluation lineage before showing a final score. Results cannot choose a new model or feature set.

In [ ]:
from nfl_trajectory.final_protocol import digest
from nfl_trajectory.final_results import load_final_results
from nfl_trajectory.runtime import sha256

final_fit = json.loads((PUBLISHED / "final_fit.json").read_text())
assert digest(final_fit["provenance"]) == final_fit["source_signature"]
for source_path, expected_hash in final_fit["provenance"]["implementation"].items():
    assert sha256(ROOT / source_path) == expected_hash
fit_table = pd.DataFrame(final_fit["profiles"])
display(
    fit_table[
        [
            "profile",
            "training_rows",
            "fitted_features",
            "active_features",
            "portable_max_absolute_difference",
        ]
    ]
)
final_results = load_final_results(ROOT)
if final_results is None:
    display(
        Markdown(
            "**Final fitting and the organizer gateway are verified. "
            "The reserved-holdout score has not yet been published.** "
            "The development scores above remain research evidence; "
            "they are not scores from the new final models."
        )
    )
else:
    heldout = final_results["evaluation"]
    metrics = pd.DataFrame(heldout["metrics"])
    display(
        Markdown(
            f"**Sealed holdout: {heldout['games']} later games and "
            f"{heldout['rows']:,} forecast rows.** "
            "The primary model was fixed before these outcomes were read."
        )
    )
    display(
        metrics[
            [
                "scenario",
                "coordinate_rmse_yards",
                "coordinate_rmse_95_interval",
                "ade_frame_weighted_yards",
                "fde_trajectory_weighted_yards",
                "p95_displacement_yards",
            ]
        ]
    )
    slices = pd.DataFrame(heldout["slices"])
    horizon = slices.loc[slices.group.eq("forecast_second")].copy()
    horizon["forecast_second"] = horizon.value.astype(int)
    horizon = horizon.sort_values("forecast_second")
    display(
        px.line(
            horizon,
            x="forecast_second",
            y="coordinate_rmse_yards",
            markers=True,
            hover_data=["rows"],
            labels={
                "forecast_second": "Forecast second (ceiling)",
                "coordinate_rmse_yards": "Coordinate RMSE (yards)",
            },
            title="Where does error grow along the forecast horizon?",
        )
    )
    roles = slices.loc[slices.group.eq("player_role")]
    display(roles[["value", "rows", "coordinate_rmse_yards", "fde_trajectory_weighted_yards"]])
    figure, axis = plt.subplots(figsize=(8, 3.5))
    axis.plot(horizon.forecast_second, horizon.coordinate_rmse_yards, marker="o")
    axis.set(
        xlabel="Forecast second (ceiling)",
        ylabel="Coordinate RMSE (yards)",
        title="Reserved games: error over the forecast horizon",
        ylim=(0, None),
    )
    static(figure)
    display(Markdown(heldout["limitations"]))

### Error concentration and temporal stability

The following is a descriptive audit after evaluation. Every rare and difficult row remains in the official score. Horizon bins contain different trajectory cohorts: the shrinking sample sizes must accompany any error trend. Baseline gains here combine features and estimator changes; the controlled feature-attribution experiment remains the development comparison above.

In [ ]:
from nfl_trajectory.final_diagnostics import load_diagnostics

if final_results is not None:
    diagnosis = load_diagnostics(ROOT)
    if diagnosis is None:
        raise ValueError("Final error-concentration evidence must be published with the results.")
    support = pd.DataFrame(diagnosis["forecast_bins"])
    display(support)
    display(pd.DataFrame(diagnosis["reference_comparisons"]))
    display(slices.loc[slices.group.eq("week"), ["value", "rows", "coordinate_rmse_yards"]])
    late = support.loc[support.forecast_second.gt(3)]
    display(
        Markdown(
            f"Beyond three seconds: **{late.rows.sum():,} rows "
            f"({late.row_share.sum():.2%}) contribute "
            f"{late.squared_error_share.sum():.2%} of squared error**. "
            f"The largest-error play contributes "
            f"{diagnosis['top_play_squared_error_share']:.2%}; the largest-error game "
            f"contributes {diagnosis['top_game_squared_error_share']:.2%}. "
            "These observations remain in the reported metric and bootstrap interval."
        )
    )
    comparison = support.melt(
        id_vars=["forecast_second", "rows", "games", "trajectories"],
        value_vars=["row_share", "squared_error_share"],
        var_name="quantity",
        value_name="share",
    )
    concentration = px.bar(
        comparison,
        x="forecast_second",
        y="share",
        color="quantity",
        barmode="group",
        hover_data=["rows", "games", "trajectories"],
        labels={"forecast_second": "Forecast second (ceiling)", "share": "Share of total"},
        title="Does forecast frequency explain each horizon's contribution to error?",
    )
    concentration.update_yaxes(tickformat=".0%")
    display(concentration)
    figure, axis = plt.subplots(figsize=(8, 3.5))
    axis.bar(
        support.forecast_second - 0.18, 100 * support.row_share, width=0.36, label="Forecast rows"
    )
    axis.bar(
        support.forecast_second + 0.18,
        100 * support.squared_error_share,
        width=0.36,
        label="Squared error",
    )
    axis.set(
        xlabel="Forecast second (ceiling)",
        ylabel="Share of total (%)",
        title="Rare long forecasts contribute disproportionate error",
    )
    axis.legend()
    static(figure)
    display(Markdown(diagnosis["scope"]))

## 9 · Temporal attention challenger

The submitted tree model's **0.70090 private Kaggle score** motivated this separate experiment. A temporal convolutional encoder now reads 20 observed feature frames, and two geometry-informed attention blocks model interactions across players. A continuous-time decoder covers every requested future frame.

Inputs include landing-relative motion, synchronized receiver/passer geometry, closest-approach features, learned role/side embeddings and earlier-date player/role target encodings. Historical tables exclude the entire current date and remain frozen on development. [The model protocol](../docs/TEMPORAL_MODEL.md) documents features, provenance, categorical treatment and sources.

The comparison below uses exactly the existing **192 training games and 32 later development games**. The final epoch EMA was chosen before training; intermediate scores do not select a checkpoint. This single experiment changes architecture and features together, so it does not isolate either contribution. Inner-fold confirmation and organizer gateway validation remain necessary before promotion. No new Kaggle result is claimed.

A first attempt was stopped when decoder saturation suppressed encoder gradients. [The training-stability record](../docs/results/temporal_stability.json) preserves that failed curve and diagnosis. Final encoder and decoder normalization were added before the completed trial shown below.

The equal-blend row is a secondary exploratory check using fixed 50/50 weights, declared during training before the final score. It adds no training and requires out-of-fold confirmation. The confidence interval below compares the individual temporal model with the tree.

In [ ]:
from nfl_trajectory.runtime import sha256

temporal_path = PUBLISHED / "temporal_model.json"
temporal_comparison_path = PUBLISHED / "temporal_evaluation.json"
if temporal_path.is_file() and temporal_comparison_path.is_file():
    temporal_trial = json.loads(temporal_path.read_text())
    temporal_comparison = json.loads(temporal_comparison_path.read_text())
    assert temporal_trial["status"] == "completed"
    assert temporal_comparison["status"] == "passed"
    assert temporal_comparison["trial_report_sha256"] == sha256(temporal_path)
    assert temporal_trial["development_rows"] == temporal_comparison["development_rows"] == 67857
    display(
        pd.DataFrame(temporal_comparison["models"])[
            [
                "model",
                "coordinate_rmse_yards",
                "ade_frame_weighted_yards",
                "fde_trajectory_weighted_yards",
                "p95_displacement_yards",
            ]
        ]
    )
    lower, upper = temporal_comparison["paired_game_bootstrap_delta_ci95"]
    display(
        Markdown(
            f"**Temporal minus tree RMSE, paired game-bootstrap 95% interval:** "
            f"{lower:.5f} to {upper:.5f} yards. Negative values favor the temporal model. "
            f"Training completed {temporal_trial['epochs_completed']} epochs in "
            f"{temporal_trial['training_seconds'] / 60:.1f} minutes on four CPU threads."
        )
    )
    import plotly.express as px

    comparison_frame = pd.DataFrame(temporal_comparison["models"])
    comparison_frame["label"] = ["Existing tree", "Temporal attention", "Equal blend · exploratory"]
    figure = px.bar(
        comparison_frame,
        x="label",
        y="coordinate_rmse_yards",
        color="label",
        text="coordinate_rmse_yards",
        color_discrete_sequence=["#b66929", "#087e8b", "#18344a"],
        labels={"label": "", "coordinate_rmse_yards": "Coordinate RMSE (yards)"},
        title="Development comparison · 32 games, 67,857 forecast rows",
        template="plotly_white",
    )
    figure.update_traces(texttemplate="%{text:.5f}", textposition="outside")
    figure.update_layout(
        showlegend=False, height=420, margin=dict(t=65, b=45), yaxis_range=[0, 0.8]
    )
    display(figure)

    display(Image(filename=str(PUBLISHED / "temporal_diagnostics.png")))
    temporal_slices = pd.DataFrame(temporal_comparison["slices"])
    display(
        temporal_slices.loc[
            temporal_slices.dimension.eq("role"),
            ["model", "value", "rows", "coordinate_rmse_yards"],
        ]
    )
else:
    display(Markdown("No completed temporal comparison is available in this checkout."))

## Generate and download your own inference artifact

The switch stays off during review and automated publication. Enabling it verifies and exports the current local **final** model, including the independently fitted positional fallback. It provides a download and never submits to Kaggle. The research exporter remains available to reproduce its original validated experiment.

In [ ]:
import base64
import subprocess
import sys

from IPython.display import HTML

GENERATE_EXPORT = False
if GENERATE_EXPORT:
    local_summary = ROOT / "artifacts/final/models/summary.json"
    if not local_summary.is_file():
        raise FileNotFoundError("Verified local final-model artifacts are required for an export.")
    subprocess.run(
        [sys.executable, str(ROOT / "kaggle/final_export.py")],
        cwd=ROOT,
        check=True,
    )
    export_path = ROOT / "artifacts/kaggle/submission.ipynb"
    payload = base64.b64encode(export_path.read_bytes()).decode("ascii")
    display(
        HTML(
            '<a download="submission.ipynb" href="data:application/x-ipynb+json;base64,'
            + payload
            + '">Download the inference notebook you generated</a>'
        )
    )
else:
    display(
        Markdown(
            "**Export is off.** Set `GENERATE_EXPORT = True` "
            "to build and download your own artifact."
        )
    )